In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

In [2]:
groundtruth = pd.read_csv("Noise_GroundTruth_and_TrainingLoss.csv")

foif = pd.read_csv("NoisyLabel_FOIF_Scores.csv")

tracin = pd.read_csv("NoisyLabel_TracIn_Scores.csv")

In [3]:
foif = foif.rename(columns={"Score":"FOIF_Score"})
tracin = tracin.rename(columns={"Score":"TracIn_Score"})

merged = (
    groundtruth
    .merge(
        foif[["Train_ID","FOIF_Score"]],
        on="Train_ID"
    )
    .merge(
        tracin[["Train_ID","TracIn_Score"]],
        on="Train_ID"
    )
)

print(merged.head())

   Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss  FOIF_Score  \
0      5018            1            1         0       0.324082    0.191415   
1     10999            1            0         1       1.027546   -0.157258   
2      3146            1            0         1       1.837602   -1.040044   
3      9147            0            0         0       0.262875    0.102674   
4     12955            1            1         0       0.520085    0.186383   

   TracIn_Score  
0      0.030606  
1     -0.050818  
2     -0.115014  
3     -0.000060  
4      0.024627  


In [4]:
random_baseline_seed = 42

rng = np.random.default_rng(random_baseline_seed)

merged["Random_Score"] = rng.random(len(merged))
# merged[["Train_ID", "Random_Score"]].to_csv(
#     "Random_Ranking.csv",
#     index=False
# )
print(merged)

      Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss  FOIF_Score  \
0         5018            1            1         0       0.324082    0.191415   
1        10999            1            0         1       1.027546   -0.157258   
2         3146            1            0         1       1.837602   -1.040044   
3         9147            0            0         0       0.262875    0.102674   
4        12955            1            1         0       0.520085    0.186383   
...        ...          ...          ...       ...            ...         ...   
7995     13382            0            0         0       0.135412    0.172427   
7996     13339            0            0         0       0.126330    0.125705   
7997     15658            0            0         0       0.392910   -0.200291   
7998     12979            0            0         0       0.273940    0.140438   
7999      9910            1            1         0       0.882032    0.131435   

      TracIn_Score  Random_

In [5]:
print(
    merged.groupby("is_noisy")[
        "FOIF_Score"
    ].mean()
)

print(
    merged.groupby("is_noisy")[
        "TracIn_Score"
    ].mean()
)

is_noisy
0    0.154775
1   -0.617654
Name: FOIF_Score, dtype: float64
is_noisy
0    0.006894
1   -0.025217
Name: TracIn_Score, dtype: float64


In [6]:
def precision_recall_at_k(
    y_true,
    scores,
    k_percentage,
    retrieve="top"
):
    """
    Parameters
    ----------
    y_true : binary ground truth
    scores : ranking scores
    k_percentage : e.g. 0.1 for Top/Bottom 10%
    retrieve : "top" or "bottom"
    """

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = int(len(scores) * k_percentage)

    if retrieve == "top":
        selected = np.argsort(scores)[::-1][:k]

    elif retrieve == "bottom":
        selected = np.argsort(scores)[:k]

    else:
        raise ValueError("retrieve must be 'top' or 'bottom'")

    tp = y_true[selected].sum()

    precision = tp / k
    recall = tp / y_true.sum()

    return precision, recall

In [7]:
results = []

methods = {

    "Random":
    {
        "score": merged["Random_Score"],
        "retrieve": "bottom"
    },

    "Training Loss":
    {
        "score": merged["Training_Loss"],
        "retrieve": "top"
    },

    "FOIF":
    {
        "score": merged["FOIF_Score"],
        "retrieve": "bottom"
    },

    "TracIn":
    {
        "score": merged["TracIn_Score"],
        "retrieve": "bottom"
    }
}

In [8]:
for method, info in methods.items():

    scores = info["score"]

    retrieve = info["retrieve"]

    # AUPRC
    #
    # sklearn assumes larger score = positive
    #
    # Therefore we only negate FOIF/TracIn
    if retrieve == "bottom":
        auprc = average_precision_score(
            merged["is_noisy"],
            -scores
        )
    else:
        auprc = average_precision_score(
            merged["is_noisy"],
            scores
        )

    row = {

        "Method": method,

        "AUPRC": auprc
    }

    for p in [0.10,0.20,0.30]:

        precision, recall = precision_recall_at_k(

            merged["is_noisy"],

            scores,

            p,

            retrieve
        )

        row[f"Precision@{int(p*100)}"] = precision

        row[f"Recall@{int(p*100)}"] = recall

    results.append(row)

In [9]:
result_df = pd.DataFrame(results)

print(result_df.round(4))

          Method   AUPRC  Precision@10  Recall@10  Precision@20  Recall@20  \
0         Random  0.2050        0.2125     0.1062        0.2169     0.2169   
1  Training Loss  0.9511        0.9725     0.4862        0.9194     0.9194   
2           FOIF  0.8735        0.9325     0.4662        0.8538     0.8538   
3         TracIn  0.6005        0.7325     0.3662        0.5950     0.5950   

   Precision@30  Recall@30  
0        0.2042     0.3062  
1        0.6571     0.9856  
2        0.6179     0.9269  
3        0.4629     0.6944  
